Create Database

In [0]:
spark.sql("CREATE DATABASE IF NOT EXISTS fraud")

In [0]:
import random
from pyspark.sql.types import *
from pyspark.sql.functions import current_timestamp

In [0]:
locations = ["London","Manchester","New York","Dubai","Berlin"]
merchants = ["Amazon","Tesco","Apple Store","Electronics Hub","Walmart"]

rows = []

for i in range(10000):

    transaction_id = f"T{i}"
    customer_id = f"C{random.randint(1,500)}"
    amount = round(random.uniform(5,7000),2)
    location = random.choice(locations)
    merchant = random.choice(merchants)
    device_id = random.randint(1,50)
    ip_address = random.randint(1,255)

    rows.append((transaction_id,customer_id,amount,location,merchant,device_id,ip_address))

In [0]:
schema = StructType([
StructField("transaction_id",StringType()),
StructField("customer_id",StringType()),
StructField("amount",DoubleType()),
StructField("location",StringType()),
StructField("merchant",StringType()),
StructField("device_id",IntegerType()),
StructField("ip_address",IntegerType())
])

df = spark.createDataFrame(rows,schema)

df = df.withColumn("timestamp",current_timestamp())

In [0]:
from pyspark.sql.functions import when

df = df.withColumn(
    "risk_level",
    when(df.amount > 5000,"HIGH")
    .when(df.amount > 2500,"MEDIUM")
    .otherwise("LOW")
)

In [0]:
df.write.format("delta").mode("overwrite").saveAsTable("fraud.transactions")

In [0]:
spark.sql("SELECT * FROM fraud.transactions LIMIT 20").display()